# OS Tutor — QLoRA Fine-Tuning v3 (grounding-consistent, refusal-aware)

**Model:** Qwen2.5-1.5B-Instruct · **Method:** QLoRA (Unsloth + TRL) · **GPU:** Colab T4 (free tier is enough)

> **Runtime → Change runtime type → T4 GPU**, then *Run all*. Everything is saved to Google Drive; a disconnect only costs the seed that was running. First time? Set `PRESET = 'smoke'` (≈10 min) to check the pipeline.

### What the v2 run (3 seeds, leak-free split) taught us
| v2 finding | evidence | v3 response |
|---|---|---|
| **The data contradicts its own system prompt.** The prompt says *"answer ONLY from the passage, else `OUT_OF_CONTEXT`"*, but ≈ half of the `recall` references (18 / 35 in the test set) are **not in their passage**, and the reasoning categories never are | answer-word overlap with the passage (`support`) | **Grounding policy** (§2): ungrounded `recall` rows are **relabelled `OUT_OF_CONTEXT`** (the passage really does not answer them) instead of teaching "answer anyway". Rows written to `relabeled_for_review.csv` so you can audit them |
| Metrics mix *grounding* with *memorised knowledge* | F1 on ungrounded rows measures what the 1.5 B model remembers | Every metric is also reported **by groundedness bucket**: `in_passage`, `beyond_passage`, `computed`, `refusal` |
| Refusal not learned (best seed refused 0 / 5) | refusal F1 0.33 ± 0.58 | Relabelled rows + off-topic rows + hard negatives (built *inside* each split); **refusal-aware test set** |
| Early stopping chose step 11 (⅓ epoch) | val loss dominated by long answers | **No early stopping**: ½-epoch checkpoints, chosen on **validation generation quality** (F1 + refusal F1 − false-refusal) |
| No gain under the deployed prompt (F1 0.346 vs base 0.359) | `rag/qwen_chain.py` uses another system prompt | **Prompt mix**: 50 % of training rows use the deployed prompt |
| One-word "Yes"/"No" answers for edge cases | worst-F1 rows | New **`TooShort`** metric (≤ 2 words when the reference has ≥ 15) |
| Token-F1 punishes paraphrase | one reference | **Semantic similarity** (bge-small) added |
| Ablations never ran | empty §11 | **`PRESET`** switch (includes grounding / reasoning ablations) |

Kept from v2: group-by-passage split (zero passage overlap), answer-only loss, un-tuned base-model baseline, 3 seeds, paired bootstrap, IEEE figures.

> **Comparability:** the v3 split groups rows by passage *and* question (40 questions occur twice in the data), so it is **not** the v2 test set — never compare v3 numbers with v2 numbers across runs. Compare *within* a run (LoRA vs. its own base model) and across `PRESET`s of this notebook (e.g. `abl_keep_ungrounded` = v2-style data on the v3 split).
> **Honest caveat:** relabelled and synthetic refusal rows are data engineering, not new textbook knowledge — say so in the paper.

## 1 · Setup

In [ ]:
# Installs Unsloth + datasets (~2-3 min on a fresh runtime). No restart needed on a FRESH runtime;
# if you already imported torch/transformers in this runtime, do Runtime > Restart session first.
!pip install -q --upgrade unsloth datasets sentence-transformers

In [ ]:
import unsloth  # noqa: F401  -- must be imported BEFORE transformers / trl / peft
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

import os, sys, gc, json, re, time, math, random, shutil, inspect, platform, warnings, collections
from importlib import metadata
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer, EarlyStoppingCallback, set_seed
from trl import SFTTrainer, SFTConfig
from peft import set_peft_model_state_dict
from safetensors.torch import load_file
import glob

warnings.filterwarnings('ignore', category=FutureWarning)
assert torch.cuda.is_available(), 'No GPU found. Runtime > Change runtime type > T4 GPU'

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s); set_seed(s)

def pkg(n):
    try: return metadata.version(n)
    except Exception: return 'n/a'

ENV = {'python': sys.version.split()[0], 'platform': platform.platform(),
       'gpu': torch.cuda.get_device_name(0), 'vram_gb': round(torch.cuda.get_device_properties(0).total_memory/1e9, 1),
       **{p: pkg(p) for p in ['torch','transformers','trl','peft','unsloth','datasets','bitsandbytes','accelerate']}}
for k, v in ENV.items(): print(f'{k:13s}: {v}')

In [ ]:
# ─────────────────────────── CONFIG (edit here) ───────────────────────────
IN_COLAB = 'google.colab' in sys.modules

PRESET = 'main'      # <- the ONLY line to change for an ablation (see PRESETS), then Run all
PRESETS = {
    'main':               {},
    'abl_no_aug':         {'AUG_OFFTOPIC': False, 'AUG_HARDNEG': False},     # does refusal augmentation matter?
    'abl_no_promptmix':   {'PROMPT_MIX': 0.0},                               # does prompt mixing matter?
    'abl_select_by_loss': {'SELECT_BY': 'val_loss'},                         # v2-style checkpoint choice, for comparison
    'abl_fullseq_loss':   {'TRAIN_ON_RESPONSES_ONLY': False},
    'abl_r8':             {'LORA_R': 8,  'LORA_ALPHA': 16},
    'abl_r32':            {'LORA_R': 32, 'LORA_ALPHA': 64},
    'abl_keep_ungrounded':{'GROUNDING_POLICY': 'keep'},                       # = v2 data: comparable with v2 results
    'abl_drop_ungrounded':{'GROUNDING_POLICY': 'drop'},
    'abl_drop_reasoning': {'REASONING_POLICY': 'drop'},                      # remove analytical/conceptual/comparative/edge rows
    'smoke':              {'SEEDS': [42], 'EPOCHS': 1},                      # ~10 min pipeline check
}

CFG = dict(
    MODEL_ID     = 'Qwen/Qwen2.5-1.5B-Instruct',
    MAX_SEQ_LEN  = 1024,
    # LoRA
    LORA_R = 16, LORA_ALPHA = 32, LORA_DROPOUT = 0.05,
    TARGET_MODULES = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    # optimisation (no early stopping: we keep checkpoints and pick one by generation quality)
    EPOCHS = 3, BATCH_SIZE = 2, GRAD_ACCUM = 8, LR = 2e-4, WEIGHT_DECAY = 0.01, WARMUP_RATIO = 0.05,
    CKPTS_PER_EPOCH = 2,
    SEEDS = [42, 43, 44],             # >=3 seeds for mean +- std
    # data
    DATA_FILES = {'train': 'dataset_v2_train.csv', 'val': 'dataset_v2_val.csv'},
    SPLIT_SEED = 42, VAL_FRAC = 0.12, TEST_FRAC = 0.12,      # NOTE: v3 groups by passage AND question, so the split differs from v2
    TRAIN_ON_RESPONSES_ONLY = True,
    OVERSAMPLE_CATS = ['comparative','analytical','negative_edge_case','applied_conceptual'], OVERSAMPLE_FACTOR = 1,
    FILTER_UNGROUNDED = False, FILTER_CATS = ['recall'], GROUND_THRESH = 0.30,
    # refusal-aware augmentation (built separately inside each split -> no leakage)
    AUG_OFFTOPIC = True,  AUG_OFF_K = 1,                     # each off-topic question paired with K different passages
    AUG_HARDNEG  = True,  HARDNEG_N = {'train': 40, 'val': 15, 'test': 25},
    EXTRA_OOC_FILE = 'ooc_extra.csv',                        # optional: CSV with a `question` column of YOUR off-topic questions
    EXTERNAL_DATA = 'dataset_v3.csv',     # optional re-grounded dataset (same columns as dataset_v2_*.csv); used instead of DATA_FILES
    GROUNDING_POLICY = 'relabel',         # recall rows whose answer is NOT in the passage: 'relabel' (-> OUT_OF_CONTEXT) | 'drop' | 'keep'
    GROUNDING_CATS = ['recall'], GROUND_THRESH_REL = 0.25,
    REASONING_POLICY = 'keep',            # analytical / applied_conceptual / comparative / negative_edge_case: 'keep' | 'drop'
    MAX_OOC_TRAIN_FRAC = 0.20,            # cap refusal rows at this share of TRAIN (None = no cap); val/test keep all
    PROMPT_MIX = 0.5,                                        # share of TRAIN rows that use the deployed system prompt
    # checkpoint selection
    SELECT_BY = 'generation',                                # 'generation' | 'val_loss'
    SEL_N_OTHER = 40, SEL_MAX_NEW = 256,
    SEL_W = {'F1': 0.5, 'Refusal_F1': 0.5, 'FalseRefusal': 0.25},   # score = .5*F1 + .5*RefusalF1 - .25*FalseRefusal
    # evaluation
    GEN_MAX_NEW_TOKENS = 400, GEN_BATCH = 8, EVAL_DEPLOY_PROMPT = True,
    KEEP_CHECKPOINTS = False, USE_DRIVE = True,
)
CFG.update(PRESETS[PRESET]); CFG['EXPERIMENT'] = f'f3_{PRESET}'; CFG['PRESET'] = PRESET

DEPLOY_SYSTEM = (
    "You are a concise Operating Systems tutor based on the Galvin textbook. "
    "Answer DIRECTLY and SPECIFICALLY. Do NOT ramble, speculate, or go off-topic. "
    "If textbook passages are provided, base your answer strictly on them. "
    "Keep answers short: 2-5 sentences maximum unless a step-by-step solution is needed. "
    "For numerical problems, show only the key steps and the final answer. "
    "NEVER invent figures, page numbers, or facts not in the context. "
    "NEVER ask follow-up questions back to the student."
)

if IN_COLAB and CFG['USE_DRIVE']:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/os_tutor_ft3'
else:
    WORK_DIR = './ft3_runs'
RUN_DIR = os.path.join(WORK_DIR, CFG['EXPERIMENT'])
os.makedirs(os.path.join(RUN_DIR, 'figures'), exist_ok=True)
json.dump(CFG, open(os.path.join(RUN_DIR, 'config.json'), 'w'), indent=2)
print('Preset:', PRESET, '| run directory:', os.path.abspath(RUN_DIR))

## 2 · Data: load, audit, **ground**, split

The audit is what a reviewer will ask for. It answers **(a)** does a row's answer actually come from its passage (the system prompt says it must), and **(b)** do val/test passages also appear in training?

**Grounding policy (new).** A `recall` row whose reference answer shares < 25 % of its content words with its passage teaches the model *"answer even when the passage does not contain the answer"* — the opposite of the refusal we want. Default: relabel those rows `OUT_OF_CONTEXT` (they are exactly the "near-topic but unanswerable" cases). The relabelled rows are saved to `relabeled_for_review.csv` — **skim it**: the overlap test is lexical, so a few wordy-but-grounded answers may be wrongly relabelled (raise `GROUND_THRESH_REL` guard or switch to `'drop'` if you see many).
Reasoning categories (analytical, conceptual, comparative, edge-case) are never in the passage by design; they are kept by default and reported in their own bucket. For best quality supply `dataset_v3.csv` with passages re-retrieved from the real textbook index.

In [ ]:
def find_file(name):
    for d in [WORK_DIR, '.', '/content']:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    return None

ext = find_file(CFG['EXTERNAL_DATA']) if CFG.get('EXTERNAL_DATA') else None
paths = {'external': ext} if ext else {k: find_file(v) for k, v in CFG['DATA_FILES'].items()}
print('Using', ('EXTERNAL re-grounded dataset: ' + ext) if ext else 'dataset_v2 train + val')
def ensure_uploaded(names):
    missing = [n for n in names if find_file(n) is None]
    if missing and IN_COLAB:
        from google.colab import files
        print('>>> Please upload these files now (use the file picker below):', missing)
        up = files.upload()                                    # lands in /content
        for n in up:
            if CFG['USE_DRIVE']:                               # keep a copy on Drive -> next session needs no upload
                shutil.copy(n, os.path.join(WORK_DIR, n)); print('   saved a copy to', os.path.join(WORK_DIR, n))
    still = [n for n in names if find_file(n) is None]
    if still: raise FileNotFoundError(f'Missing data files: {still}. Put them in {WORK_DIR} or the working directory.')

if any(p is None for p in paths.values()):
    ensure_uploaded(list(CFG['DATA_FILES'].values()))
    paths = {k: find_file(v) for k, v in CFG['DATA_FILES'].items()}

raw = pd.concat([pd.read_csv(p).assign(orig_split=k) for k, p in paths.items()], ignore_index=True)
need = {'category','difficulty','system','prompt','completion'}
assert need <= set(raw.columns), f'missing columns: {need - set(raw.columns)}'
raw = raw.dropna(subset=['system','prompt','completion']).reset_index(drop=True)
raw['orig_completion'] = raw.completion

CTX_RE = re.compile(r'Textbook passages:\s*(.*?)\s*Question:\s*(.*)\Z', re.S)
def parse_prompt(p):
    m = CTX_RE.search(p)
    return (m.group(1).strip(), m.group(2).strip()) if m else ('', p.strip())
parsed = [parse_prompt(p) for p in raw.prompt]
raw['context']  = [a for a, _ in parsed]
raw['question'] = [b for _, b in parsed]

WORD4 = re.compile(r'[a-z]{4,}')
def support(answer, context):
    # fraction of the answer's content words (>=4 letters) that occur in the passage -- lexical groundedness proxy
    a = set(WORD4.findall(answer.lower())); c = set(WORD4.findall(context.lower()))
    return len(a & c) / max(1, len(a))
raw['support'] = [support(a, c) for a, c in zip(raw.completion, raw.context)]
raw['is_ooc']  = raw.completion.str.strip().eq('OUT_OF_CONTEXT')

SYSTEM_PROMPT = raw.system.mode()[0]

# ── Audit ────────────────────────────────────────────────────────────────
tr_ctx = set(raw.loc[raw.orig_split=='train','context'])
va = raw[raw.orig_split=='val']
audit = {
    'rows': len(raw), 'unique_passages': int(raw.context.nunique()),
    'unparsed_prompts': int((raw.context=='').sum()),
    'duplicate_rows': int(raw.duplicated(['system','prompt','completion']).sum()),
    'distinct_system_prompts': int(raw.system.nunique()),
    'original_val_rows_sharing_passage_with_train': int(va.context.isin(tr_ctx).sum()),
    'original_val_rows': int(len(va)),
    'ooc_rows': int((raw.category=='out_of_context').sum()),
    'ooc_unique_questions': int(raw.loc[raw.category=='out_of_context','question'].nunique()),
    'ooc_rows_wrong_category': int((raw.is_ooc & (raw.category!='out_of_context')).sum()),
}
print(json.dumps(audit, indent=2))
print('\nGroundedness proxy (share of answer words found in the passage), by category:')
g = raw.groupby('category').support.agg(['count','mean','median']).round(2)
g['ungrounded(<%.2f)' % CFG['GROUND_THRESH']] = raw.assign(u=raw.support < CFG['GROUND_THRESH']).groupby('category').u.sum()
print(g.to_string())
json.dump(audit, open(os.path.join(RUN_DIR,'data_audit.json'),'w'), indent=2)

# ── Grounding / consistency policy (v3) ────────────────────────────────────
REASONING = ['analytical', 'applied_conceptual', 'comparative', 'negative_edge_case']
raw['relabeled'] = False
ung = raw.category.isin(CFG['GROUNDING_CATS']) & (raw.support < CFG['GROUND_THRESH_REL']) & ~raw.is_ooc
print(f"\nGrounding policy = {CFG['GROUNDING_POLICY']} | ungrounded rows in {CFG['GROUNDING_CATS']} (support < {CFG['GROUND_THRESH_REL']}): {int(ung.sum())}")
if CFG['GROUNDING_POLICY'] == 'relabel' and ung.any():
    raw[ung].assign(new_label='OUT_OF_CONTEXT')[['category', 'question', 'orig_completion', 'support', 'new_label']] \
        .to_csv(os.path.join(RUN_DIR, 'relabeled_for_review.csv'), index=False)
    raw.loc[ung, 'completion'] = 'OUT_OF_CONTEXT'; raw.loc[ung, 'category'] = 'ooc_ungrounded'
    raw.loc[ung, 'is_ooc'] = True;                 raw.loc[ung, 'support'] = 0.0; raw.loc[ung, 'relabeled'] = True
    print('Relabelled rows saved to relabeled_for_review.csv. Sample (question -> what the old answer said):')
    for r in pd.read_csv(os.path.join(RUN_DIR, 'relabeled_for_review.csv')).head(4).itertuples():
        print('  *', r.question[:90].replace('\n', ' '), '->', str(r.orig_completion)[:90].replace('\n', ' '))
elif CFG['GROUNDING_POLICY'] == 'drop':
    raw = raw[~ung]
if CFG['REASONING_POLICY'] == 'drop':
    print('Dropping reasoning categories:', int(raw.category.isin(REASONING).sum()), 'rows'); raw = raw[~raw.category.isin(REASONING)]
raw = raw.reset_index(drop=True)

def bucket(r):
    if r.is_ooc: return 'refusal'
    if r.category == 'applied_numerical': return 'computed'            # derived from the question's numbers
    return 'in_passage' if r.support >= 0.5 else 'beyond_passage'
raw['gbucket'] = [bucket(r) for r in raw.itertuples()]
# split-group key = connected components of "shares a passage" (answerable rows only) or "shares a question" (all rows),
# so neither a passage nor a question can leak across train / val / test (40 questions occur twice in the data)
_par = list(range(len(raw)))
def _find(x):
    while _par[x] != x: _par[x] = _par[_par[x]]; x = _par[x]
    return x
def _union(a, b): _par[_find(a)] = _find(b)
_ctx_first, _q_first = {}, {}
for i, r in enumerate(raw.itertuples()):
    if not r.is_ooc: _union(i, _ctx_first.setdefault(r.context, i))
    _union(i, _q_first.setdefault(r.question, i))
raw['gkey'] = ['G%d' % _find(i) for i in range(len(raw))]
print(f"Split groups: {raw.gkey.nunique()} (largest = {raw.gkey.value_counts().iloc[0]} rows)")
print('\nGroundedness buckets (all rows):'); print(pd.crosstab(raw.category, raw.gbucket).to_string())

In [ ]:
def group_stratified_split(df, val_frac, test_frac, seed):
    # Assign whole passages (groups) to train/val/test so no passage is shared, while keeping the
    # per-category proportions close to the targets (greedy on the relative deficit).
    rng = np.random.RandomState(seed)
    cats = sorted(df.category.unique())
    tot = df.category.value_counts().to_dict()
    frac = {'val': val_frac, 'test': test_frac, 'train': 1 - val_frac - test_frac}
    target = {s: {c: tot[c]*frac[s] for c in cats} for s in frac}
    cur = {s: {c: 0 for c in cats} for s in frac}
    items = list(df.groupby('gkey').indices.items())
    rng.shuffle(items); items.sort(key=lambda kv: -len(kv[1]))     # big groups first, random among ties
    assign = np.empty(len(df), dtype=object)
    for _, idx in items:
        cnt = df.category.iloc[idx].value_counts()
        best, best_s = None, -1e18
        for s in ('val', 'test', 'train'):
            score = sum(n * (target[s][c] - cur[s][c]) / max(target[s][c], 1e-9) for c, n in cnt.items())
            if score > best_s: best, best_s = s, score
        assign[idx] = best
        for c, n in cnt.items(): cur[best][c] += n
    return assign

split_file = os.path.join(RUN_DIR, f"splits_s{CFG['SPLIT_SEED']}_v{CFG['VAL_FRAC']}_t{CFG['TEST_FRAC']}.csv")
if os.path.exists(split_file):
    sp = pd.read_csv(split_file); assert len(sp) == len(raw)
    raw['split'] = sp['split'].values
    print('Loaded existing split:', split_file)
else:
    raw['split'] = group_stratified_split(raw, CFG['VAL_FRAC'], CFG['TEST_FRAC'], CFG['SPLIT_SEED'])
    raw[['category','question','split']].to_csv(split_file, index=False)

# ── Leakage checks ───────────────────────────────────────────────────────
ctx_by = {s: set(raw.loc[(raw.split==s) & ~raw.is_ooc,'context']) for s in ['train','val','test']}
assert not (ctx_by['train'] & ctx_by['val']) and not (ctx_by['train'] & ctx_by['test']) and not (ctx_by['val'] & ctx_by['test'])
_qs = raw.groupby('question').split.nunique()
assert int((_qs > 1).sum()) == 0, f'{int((_qs > 1).sum())} questions occur in more than one split'
print('Passage overlap between splits (answerable rows): 0  |  question overlap: 0  (group split OK)')

def qset(q): return set(re.findall(r'\w+', q.lower()))
def near_dup_train_rows(thr=0.9):
    held = [(i, qset(q)) for i, q in zip(raw.index[raw.split!='train'], raw.loc[raw.split!='train','question'])]
    bad = []
    for i, q in zip(raw.index[raw.split=='train'], raw.loc[raw.split=='train','question']):
        a = qset(q)
        if any(len(a & b)/max(1, len(a | b)) >= thr for _, b in held): bad.append(i)
    return bad
nd = near_dup_train_rows()
print(f'Train rows with a near-duplicate question (Jaccard>=0.9) in val/test: {len(nd)} -> dropped')
raw.loc[nd, 'split'] = 'dropped_neardup'

tab = pd.crosstab(raw.category, raw.split)
tab['total'] = tab.sum(axis=1)
print('\nCategory x split:'); print(tab.to_string())
for s in ('val', 'test'):
    miss = [c for c in raw.category.unique() if ((raw.split==s) & (raw.category==c)).sum() == 0 and (raw.category==c).sum() >= 5]
    if miss: print(f'WARNING: {s} has no rows for {miss}')

## 2b · Refusal-aware augmentation (v3)
v2 learned to refuse from only **14 distinct off-topic questions**. Two kinds of extra `OUT_OF_CONTEXT` rows are added, **separately inside train / val / test** (a question never appears in two splits):
1. **Off-topic** general-knowledge questions (question groups are assigned to splits; each is paired with `AUG_OFF_K` different passages).
2. **Hard negatives**: a real OS question from the split, paired with a *different* passage that does not cover it (lexical overlap ≤ 5 %). This teaches "check the passage", not "detect non-OS questions".

Add your own off-topic or near-topic questions in `ooc_extra.csv` (column `question`) — they are merged into the pool.

In [ ]:
OFFTOPIC_QUESTIONS = [
 "How do you bake a sourdough bread?", "What is the capital of Australia?", "Who painted the Mona Lisa?",
 "What is the chemical symbol for gold?", "How many players are on a football (soccer) team?", "What is the tallest mountain in the world?",
 "Who wrote the novel Pride and Prejudice?", "What is the boiling point of ethanol?", "How do vaccines train the immune system?",
 "What is the largest planet in the solar system?", "How do I tie a Windsor knot?", "What year did the Berlin Wall fall?",
 "What is the main ingredient in guacamole?", "Who discovered penicillin?", "What causes the seasons on Earth?",
 "How many bones are in the adult human body?", "What is the longest river in Africa?", "Who composed the Four Seasons?",
 "What is the speed of sound in air?", "How does photosynthesis work?", "What is the currency of Switzerland?",
 "Who was the first person to walk on the moon?", "How do you make a classic margarita?", "What is the plural of 'cactus'?",
 "What language is spoken in Brazil?", "How long should I boil an egg for a soft yolk?", "What is the GDP of France?",
 "Which element has the atomic number 6?", "How do bees make honey?", "What is the rule for offside in football?",
 "What does a cardiologist treat?", "Who is the author of 'The Odyssey'?", "How do I change a flat bicycle tire?",
 "What is the freezing point of water in Fahrenheit?", "Which ocean is the deepest?", "What is the Pythagorean theorem?",
 "What are the primary colors of light?", "How many strings does a standard violin have?", "Who won the 2010 FIFA World Cup?",
 "What is the best way to learn to swim?", "What is the population of Japan?", "How is cheese made?",
 "What is the origin of the Olympic Games?", "How does a rainbow form?", "What is the square root of 144?",
 "Who is the current secretary-general of the United Nations?", "What are the symptoms of the common cold?", "How do airplanes stay in the air?",
 "What is a good recipe for chicken curry?", "Who invented the telephone?", "What is the Great Barrier Reef?",
 "How many days are in a leap year?", "Which vitamin do we get from sunlight?", "What is the melting point of iron?",
 "Who sculpted the statue of David?", "What is the national flower of India?", "How do I write a formal cover letter?",
 "What is inflation in economics?", "Which planet is known as the Red Planet?", "How does a mortgage work?",
]
_extra = find_file(CFG['EXTRA_OOC_FILE']) if CFG['EXTRA_OOC_FILE'] else None      # optional file
if _extra:
    OFFTOPIC_QUESTIONS += [q for q in pd.read_csv(_extra)['question'].dropna().astype(str)]
    print('Loaded extra off-topic questions from', _extra)

raw['synthetic'] = False
def make_ooc_row(split, ctx, q, cat):
    return dict(category=cat, difficulty='easy', system=SYSTEM_PROMPT,
                prompt=f'Textbook passages:\n{ctx}\n\nQuestion: {q}', completion='OUT_OF_CONTEXT',
                orig_split='synthetic', context=ctx, question=q, support=0.0, is_ooc=True,
                gkey='OOC::' + q, split=split, synthetic=True, relabeled=False, gbucket='refusal', orig_completion='OUT_OF_CONTEXT')

rng_aug = np.random.RandomState(CFG['SPLIT_SEED'])
new_rows = []
ctx_pool = {s: raw[(raw.split == s) & ~raw.is_ooc].context.unique() for s in ['train', 'val', 'test']}

if CFG['AUG_OFFTOPIC']:
    seen = set(raw.loc[raw.is_ooc, 'question'])
    qs = [q for q in dict.fromkeys(OFFTOPIC_QUESTIONS) if q not in seen]
    rng_aug.shuffle(qs)
    n = len(qs); cut = {'train': qs[:int(.70*n)], 'val': qs[int(.70*n):int(.85*n)], 'test': qs[int(.85*n):]}
    for sp, qlist in cut.items():
        for q in qlist:
            for ctx in rng_aug.choice(ctx_pool[sp], size=min(CFG['AUG_OFF_K'], len(ctx_pool[sp])), replace=False):
                new_rows.append(make_ooc_row(sp, ctx, q, 'out_of_context'))

if CFG['AUG_HARDNEG']:
    donors = ['recall', 'applied_conceptual', 'analytical', 'comparative', 'negative_edge_case']
    for sp, N in CFG['HARDNEG_N'].items():
        cand = raw[(raw.split == sp) & raw.category.isin(donors)].sample(frac=1, random_state=CFG['SPLIT_SEED'])
        made = 0
        for r in cand.itertuples():
            if made >= N: break
            qw = set(WORD4.findall(r.question.lower()))
            ok = [c for c in ctx_pool[sp] if c != r.context and len(qw & set(WORD4.findall(c.lower()))) / max(1, len(qw)) <= 0.05]
            if not ok: continue
            new_rows.append(make_ooc_row(sp, ok[rng_aug.randint(len(ok))], r.question, 'ooc_hard')); made += 1
        print(f'hard negatives [{sp}]: {made}/{N}')

if new_rows:
    raw = pd.concat([raw, pd.DataFrame(new_rows)], ignore_index=True)
print('\nRows after augmentation (synthetic marked):')
print(pd.crosstab(raw.category, [raw.split, raw.synthetic]).to_string())

In [ ]:
tok_cpu = AutoTokenizer.from_pretrained(CFG['MODEL_ID'], trust_remote_code=True)
RESP_PART = '<|im_start|>assistant\n'
INST_PART = '<|im_start|>user\n'

def full_text(r):
    return tok_cpu.apply_chat_template(
        [{'role':'system','content':r.system}, {'role':'user','content':r.prompt}, {'role':'assistant','content':r.completion}],
        tokenize=False, add_generation_prompt=False)
# prompt-mix augmentation: some TRAIN rows use the deployed system prompt (train/serve robustness)
rng_mix = np.random.RandomState(CFG['SPLIT_SEED'])
mix = (raw.split == 'train') & (rng_mix.rand(len(raw)) < CFG['PROMPT_MIX'])
raw.loc[mix, 'system'] = DEPLOY_SYSTEM
print(f'Training rows using the deployed system prompt: {int(mix.sum())} / {int((raw.split=="train").sum())}')
raw['text']   = [full_text(r) for r in raw.itertuples()]
raw['n_tok']  = [len(tok_cpu(t).input_ids) for t in raw.text]
too_long = raw.n_tok > CFG['MAX_SEQ_LEN']
print(f"Rows longer than MAX_SEQ_LEN={CFG['MAX_SEQ_LEN']} tokens (dropped): {int(too_long.sum())}  | max={raw.n_tok.max()}  mean={raw.n_tok.mean():.0f}")
raw = raw[~too_long & raw.split.isin(['train','val','test'])].reset_index(drop=True)

train_df = raw[raw.split=='train'].copy()
val_df   = raw[raw.split=='val'].reset_index(drop=True)
test_df  = raw[raw.split=='test'].reset_index(drop=True)

if CFG['FILTER_UNGROUNDED']:
    bad = train_df.category.isin(CFG['FILTER_CATS']) & (train_df.support < CFG['GROUND_THRESH'])
    print(f'FILTER_UNGROUNDED: dropping {int(bad.sum())} ungrounded training rows')
    train_df = train_df[~bad]
if CFG['OVERSAMPLE_FACTOR'] > 1:
    m = train_df.category.isin(CFG['OVERSAMPLE_CATS'])
    train_df = pd.concat([train_df] + [train_df[m]] * (CFG['OVERSAMPLE_FACTOR'] - 1), ignore_index=True)
if CFG['MAX_OOC_TRAIN_FRAC']:
    _f = CFG['MAX_OOC_TRAIN_FRAC']; _o = train_df.is_ooc; _max = int(_f * (~_o).sum() / (1 - _f))
    if _o.sum() > _max:
        print(f'Capping refusal rows in TRAIN: {int(_o.sum())} -> {_max} (random, proportional across kinds)')
        train_df = pd.concat([train_df[~_o], train_df[_o].sample(n=_max, random_state=CFG['SPLIT_SEED'])])
train_df = train_df.sample(frac=1, random_state=CFG['SPLIT_SEED']).reset_index(drop=True)
print('Train composition:', train_df.gbucket.value_counts().to_dict(), f'| refusal share {train_df.is_ooc.mean():.0%}')
print(f'\nFinal sizes -> train {len(train_df)} | val {len(val_df)} | test {len(test_df)}')

## 3 · Evaluation toolkit
All metrics are computed against the reference answers on the **held-out test set** (greedy decoding, so results are deterministic).

* **EM / token-F1 / ROUGE-L** — answer overlap with the reference
* **Number recall** — share of the reference's numbers that appear in the prediction (the right metric for `applied_numerical`, where F1 over words hides arithmetic errors)
* **Refusal P / R / F1** — treating `OUT_OF_CONTEXT` as the positive class; **false-refusal rate** = in-context questions wrongly refused
* **Context support** — lexical groundedness proxy (answer words found in the passage). It is a cheap proxy, *not* a replacement for RAGAS faithfulness.
* **Answer-NLL** — mean negative log-likelihood of the reference answer tokens (perplexity = exp of it)

In [ ]:
_TOK = re.compile(r'[a-z0-9]+(?:\.[0-9]+)?')
def toks(s): return _TOK.findall(str(s).lower())
def f1_tokens(p, r):
    p, r = toks(p), toks(r)
    if not p or not r: return float(p == r)
    c = sum((collections.Counter(p) & collections.Counter(r)).values())
    if c == 0: return 0.0
    pr, rc = c/len(p), c/len(r); return 2*pr*rc/(pr+rc)
def rouge_l(p, r):
    p, r = toks(p), toks(r)
    if not p or not r: return float(p == r)
    prev = [0]*(len(r)+1)
    for a in p:
        cur = [0]
        for j, b in enumerate(r): cur.append(prev[j]+1 if a == b else max(prev[j+1], cur[j]))
        prev = cur
    l = prev[-1]
    if l == 0: return 0.0
    pr, rc = l/len(p), l/len(r); return 2*pr*rc/(pr+rc)
def exact(p, r): return float(toks(p) == toks(r))
def is_ooc(s): return str(s).strip().strip('.\'"` \n').upper() == 'OUT_OF_CONTEXT'
_NUM = re.compile(r'\d+(?:\.\d+)?')
def num_recall(p, r):
    rn = _NUM.findall(str(r))
    if not rn: return np.nan
    pn = collections.Counter(_NUM.findall(str(p)))
    hit = 0
    for n in rn:
        if pn[n] > 0: pn[n] -= 1; hit += 1
    return hit/len(rn)

try:
    from sentence_transformers import SentenceTransformer
    EMB = SentenceTransformer('BAAI/bge-small-en-v1.5', device='cpu')
except Exception as e:
    EMB = None; print('SemSim disabled:', e)
def sem_sim(preds, refs):
    if EMB is None: return [np.nan] * len(preds)
    a = EMB.encode([str(x) for x in preds], normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    b = EMB.encode([str(x) for x in refs],  normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    return (a * b).sum(1)

def score_predictions(df, preds):
    out = df[['category','difficulty','question','completion','context','synthetic','gbucket']].copy()
    out['ref_words'] = out.completion.str.split().str.len()
    out['pred'] = preds
    out['em']  = [exact(p, r) for p, r in zip(out.pred, out.completion)]
    out['f1']  = [f1_tokens(p, r) for p, r in zip(out.pred, out.completion)]
    out['rougeL'] = [rouge_l(p, r) for p, r in zip(out.pred, out.completion)]
    out['num_recall'] = [num_recall(p, r) for p, r in zip(out.pred, out.completion)]
    out['support'] = [support(p, c) for p, c in zip(out.pred, out.context)]
    out['ref_ooc']  = out.completion.map(is_ooc)
    out['pred_ooc'] = out.pred.map(is_ooc)
    out['n_words']  = out.pred.str.split().str.len()
    out['semsim']   = sem_sim(out.pred, out.completion)
    out['ooc_ok']   = (out.pred_ooc == out.ref_ooc).astype(float)      # refusal DECISION correct (answer vs refuse)
    return out

def summarize(s):
    tp = int((s.ref_ooc & s.pred_ooc).sum()); fp = int((~s.ref_ooc & s.pred_ooc).sum()); fn = int((s.ref_ooc & ~s.pred_ooc).sum())
    P = tp/(tp+fp) if tp+fp else 0.0; R = tp/(tp+fn) if tp+fn else 0.0
    return dict(n=len(s), EM=s.em.mean(), F1=s.f1.mean(), ROUGE_L=s.rougeL.mean(),
                NumRecall=s[s.category=='applied_numerical'].num_recall.mean(),
                Refusal_P=P, Refusal_R=R, Refusal_F1=(2*P*R/(P+R) if P+R else 0.0),
                FalseRefusal=float((~s.ref_ooc & s.pred_ooc).sum()/max(1, (~s.ref_ooc).sum())),
                ContextSupport=s[~s.ref_ooc].support.mean(), AvgWords=s.n_words.mean(),
                SemSim=s.semsim.mean(), DecisionAcc=s.ooc_ok.mean(),
                TooShort=float(((s.n_words <= 2) & (s.ref_words >= 15) & ~s.ref_ooc).sum() / max(1, (~s.ref_ooc).sum())))

def boot_ci(x, n=2000, seed=0, alpha=0.05):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    rng = np.random.RandomState(seed)
    m = rng.choice(x, (n, len(x))).mean(1)
    return float(np.percentile(m, 100*alpha/2)), float(np.percentile(m, 100*(1-alpha/2)))

In [ ]:
def chat_prompt(tok, system, user):
    return tok.apply_chat_template([{'role':'system','content':system}, {'role':'user','content':user}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_answers(model, tok, df, system=None, max_new=None, bs=None):
    # Greedy decoding (deterministic). `system=None` uses each row's own (training) system prompt.
    max_new = max_new or CFG['GEN_MAX_NEW_TOKENS']; bs = bs or CFG['GEN_BATCH']
    tok.padding_side = 'left'
    systems = list(system) if isinstance(system, (list, tuple)) else [system] * len(df)
    prompts = [chat_prompt(tok, sy or r.system, r.prompt) for sy, r in zip(systems, df.itertuples())]
    order = np.argsort([-len(p) for p in prompts]); outs = [None]*len(prompts)
    for i in range(0, len(order), bs):
        ids = order[i:i+bs]
        enc = tok([prompts[j] for j in ids], return_tensors='pt', padding=True, truncation=True,
                  max_length=CFG['MAX_SEQ_LEN']).to('cuda')
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True, pad_token_id=tok.pad_token_id)
        for j, g in zip(ids, gen[:, enc.input_ids.shape[1]:]):
            outs[j] = tok.decode(g, skip_special_tokens=True).strip()
    return outs

@torch.no_grad()
def answer_nll(model, tok, texts):
    # mean NLL over the ANSWER tokens only (everything after the last assistant marker)
    model.eval(); tok.padding_side = 'right'
    tot, cnt = 0.0, 0
    for t in texts:
        enc = tok(t, return_tensors='pt', truncation=True, max_length=CFG['MAX_SEQ_LEN']).to('cuda')
        k = t.rfind(RESP_PART)
        n_prefix = len(tok(t[:k+len(RESP_PART)], add_special_tokens=False).input_ids)
        labels = enc.input_ids.clone(); labels[:, :n_prefix] = -100
        logits = model(**enc).logits[:, :-1].float()
        tgt = labels[:, 1:]
        tot += F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100, reduction='sum').item()
        cnt += int((tgt != -100).sum())
    return tot / max(cnt, 1)

def free_gpu(*objs):
    for o in objs: del o
    gc.collect(); torch.cuda.empty_cache()

## 4 · Baseline — the un-tuned base model on the same test set
This isolates what fine-tuning actually adds (RQ: *does LoRA help beyond prompting?*).

In [ ]:
import hashlib
_h = hashlib.md5(('v3b|' + '||'.join(test_df.prompt + '>>' + test_df.completion) + CFG['MODEL_ID']).encode('utf8')).hexdigest()[:10]
base_dir = os.path.join(WORK_DIR, 'base_' + _h); os.makedirs(base_dir, exist_ok=True)
if os.path.exists(os.path.join(base_dir, 'metrics.json')):
    print('Base model already evaluated -> skipping')
else:
    seed_everything(0)
    model, tok = FastLanguageModel.from_pretrained(model_name=CFG['MODEL_ID'], max_seq_length=CFG['MAX_SEQ_LEN'],
                                                    load_in_4bit=True, dtype=None)
    FastLanguageModel.for_inference(model)
    res = {}
    for name, system in [('train_prompt', None)] + ([('deploy_prompt', DEPLOY_SYSTEM)] if CFG['EVAL_DEPLOY_PROMPT'] else []):
        t0 = time.time()
        preds = generate_answers(model, tok, test_df, system=system)
        sc = score_predictions(test_df, preds); sc.to_csv(os.path.join(base_dir, f'test_preds_{name}.csv'), index=False)
        res[name] = summarize(sc); res[name + '_orig'] = summarize(sc[~sc.synthetic.astype(bool)])
        print(name, 'ALL', {k: round(v, 3) for k, v in res[name].items()}, f'({time.time()-t0:.0f}s)')
    res['answer_nll_val']  = answer_nll(model, tok, list(val_df.text))
    res['answer_nll_test'] = answer_nll(model, tok, list(test_df.text))
    json.dump(res, open(os.path.join(base_dir, 'metrics.json'), 'w'), indent=2)
    free_gpu(model, tok)
base_res = json.load(open(os.path.join(base_dir, 'metrics.json')))
print('\nBase answer-NLL  val %.3f | test %.3f' % (base_res['answer_nll_val'], base_res['answer_nll_test']))

## 5 · Fine-tuning (one run per seed)
Each seed: fresh 4-bit base → LoRA → answer-only SFT with early stopping → best checkpoint restored → adapter saved → test evaluation.
Finished seeds are skipped on re-run (safe against Colab disconnects).

In [ ]:
def make_sft_config(out_dir, seed, save_steps, warmup):
    P = set(inspect.signature(SFTConfig.__init__).parameters)
    bf16 = torch.cuda.is_bf16_supported()
    kw = dict(
        output_dir=out_dir, num_train_epochs=CFG['EPOCHS'],
        per_device_train_batch_size=CFG['BATCH_SIZE'], per_device_eval_batch_size=2,
        gradient_accumulation_steps=CFG['GRAD_ACCUM'], gradient_checkpointing=True,
        optim='adamw_8bit', learning_rate=CFG['LR'], lr_scheduler_type='cosine',
        warmup_steps=warmup, weight_decay=CFG['WEIGHT_DECAY'], fp16=not bf16, bf16=bf16,
        dataset_text_field='text', packing=False, logging_steps=max(1, save_steps // 4),
        save_strategy='steps', save_steps=save_steps, save_total_limit=None, save_only_model=True,
        load_best_model_at_end=False,                         # selection is done AFTER training, on generation quality
        report_to='none', seed=seed, data_seed=seed,
    )
    kw['eval_strategy' if 'eval_strategy' in P else 'evaluation_strategy'] = 'steps'
    kw['eval_steps'] = save_steps
    kw['max_seq_length' if 'max_seq_length' in P else 'max_length'] = CFG['MAX_SEQ_LEN']
    dropped = [k for k in kw if k not in P]
    if dropped: print('  (ignored unsupported SFTConfig args:', dropped, ')')
    return SFTConfig(**{k: v for k, v in kw.items() if k in P})

def ckpt_step(path): return int(os.path.basename(path).split('-')[-1])
def load_adapter_weights(model, cdir):
    f = os.path.join(cdir, 'adapter_model.safetensors')
    set_peft_model_state_dict(model, load_file(f, device='cpu'))
def sel_score(m):
    w = CFG['SEL_W']; return w['F1'] * m['F1'] + w['Refusal_F1'] * m['Refusal_F1'] - w['FalseRefusal'] * m['FalseRefusal']
def build_selection_set():
    ooc = val_df[val_df.completion.map(is_ooc)]; oth = val_df[~val_df.completion.map(is_ooc)]
    oth = oth.sample(n=min(CFG['SEL_N_OTHER'], len(oth)), random_state=CFG['SPLIT_SEED'])
    sel = pd.concat([ooc, oth]).reset_index(drop=True)
    return sel, [None if i % 2 == 0 else DEPLOY_SYSTEM for i in range(len(sel))]    # half train prompt, half deployed prompt

def run_seed(seed):
    sd = os.path.join(RUN_DIR, f'seed_{seed}'); mfile = os.path.join(sd, 'metrics.json')
    if os.path.exists(mfile):
        print(f'[seed {seed}] already done -> skipping'); return json.load(open(mfile))
    os.makedirs(sd, exist_ok=True); seed_everything(seed); t_start = time.time()
    ckpt_root = os.path.join('/content/ckpt' if IN_COLAB else sd, f"{CFG['EXPERIMENT']}_s{seed}")
    shutil.rmtree(ckpt_root, ignore_errors=True)

    model, tok = FastLanguageModel.from_pretrained(model_name=CFG['MODEL_ID'], max_seq_length=CFG['MAX_SEQ_LEN'],
                                                    load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=CFG['LORA_R'], lora_alpha=CFG['LORA_ALPHA'], lora_dropout=CFG['LORA_DROPOUT'],
        target_modules=CFG['TARGET_MODULES'], bias='none', use_gradient_checkpointing='unsloth', random_state=seed)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())

    train_ds = Dataset.from_dict({'text': list(train_df.text)}).shuffle(seed=seed)
    val_ds   = Dataset.from_dict({'text': list(val_df.text)})
    spe = max(1, len(train_ds) // (CFG['BATCH_SIZE'] * CFG['GRAD_ACCUM']))
    save_steps = max(4, spe // CFG['CKPTS_PER_EPOCH'])
    total = spe * CFG['EPOCHS']; warmup = max(1, int(total * CFG['WARMUP_RATIO']))
    print(f'[seed {seed}] train={len(train_ds)} steps/epoch={spe} total={total} ckpt_every={save_steps} | trainable {n_train/1e6:.1f}M / {n_total/1e6:.0f}M')

    T = set(inspect.signature(SFTTrainer.__init__).parameters)
    trainer = SFTTrainer(model=model, args=make_sft_config(ckpt_root, seed, save_steps, warmup),
                         train_dataset=train_ds, eval_dataset=val_ds,
                         **({'processing_class': tok} if 'processing_class' in T else {'tokenizer': tok}))
    if CFG['TRAIN_ON_RESPONSES_ONLY']:
        trainer = train_on_responses_only(trainer, instruction_part=INST_PART, response_part=RESP_PART)
    trainer.train()
    st = trainer.state; log = st.log_history
    json.dump(log, open(os.path.join(sd, 'log_history.json'), 'w'))
    final_dir = os.path.join(ckpt_root, f'checkpoint-{st.global_step}')
    if not os.path.exists(final_dir): model.save_pretrained(final_dir)
    train_minutes = (time.time() - t_start) / 60

    # ── checkpoint selection on VALIDATION (never on test) ─────────────────────────
    FastLanguageModel.for_inference(model)
    evl = {h['step']: h['eval_loss'] for h in log if 'eval_loss' in h}
    sel_df, sel_sys = build_selection_set(); cands = []
    for c in sorted(glob.glob(os.path.join(ckpt_root, 'checkpoint-*')), key=ckpt_step):
        step = ckpt_step(c); load_adapter_weights(model, c)
        if CFG['SELECT_BY'] == 'generation':
            m = summarize(score_predictions(sel_df, generate_answers(model, tok, sel_df, system=sel_sys, max_new=CFG['SEL_MAX_NEW'])))
            cands.append(dict(step=step, score=sel_score(m), val_loss=evl.get(step), F1=m['F1'], Refusal_F1=m['Refusal_F1'], FalseRefusal=m['FalseRefusal']))
        else:
            cands.append(dict(step=step, score=-evl.get(step, 1e9), val_loss=evl.get(step), F1=None, Refusal_F1=None, FalseRefusal=None))
        print(f"   ckpt {step:4d}: score {cands[-1]['score']:.3f} | val_loss {cands[-1]['val_loss']} | F1 {cands[-1]['F1']} | refusalF1 {cands[-1]['Refusal_F1']}")
    best = max(cands, key=lambda c: (c['score'], -c['step']))
    load_adapter_weights(model, os.path.join(ckpt_root, f"checkpoint-{best['step']}"))
    adapter_dir = os.path.join(sd, 'adapter'); model.save_pretrained(adapter_dir); tok.save_pretrained(adapter_dir)
    json.dump(cands, open(os.path.join(sd, 'selection.json'), 'w'), indent=1)

    out = dict(seed=seed, best_step=best['step'], stopped_step=int(st.global_step), sel_score=best['score'],
               candidates=cands, train_minutes=round(train_minutes, 1), trainable_params=int(n_train), total_params=int(n_total),
               answer_nll_val=answer_nll(model, tok, list(val_df.text)), answer_nll_test=answer_nll(model, tok, list(test_df.text)))
    for name, system in [('train_prompt', None)] + ([('deploy_prompt', DEPLOY_SYSTEM)] if CFG['EVAL_DEPLOY_PROMPT'] else []):
        sc = score_predictions(test_df, generate_answers(model, tok, test_df, system=system))
        sc.to_csv(os.path.join(sd, f'test_preds_{name}.csv'), index=False)
        out[name] = summarize(sc); out[name + '_orig'] = summarize(sc[~sc.synthetic.astype(bool)])
    if not CFG['KEEP_CHECKPOINTS']: shutil.rmtree(ckpt_root, ignore_errors=True)
    json.dump(out, open(mfile, 'w'), indent=2)
    print(f"[seed {seed}] selected step {out['best_step']}/{st.global_step} (score {best['score']:.3f}) | "
          f"test F1(orig) {out['train_prompt_orig']['F1']:.3f} | refusal F1 {out['train_prompt']['Refusal_F1']:.3f} | {(time.time()-t_start)/60:.1f} min")
    free_gpu(model, tok, trainer)
    return out

all_runs = {s: run_seed(s) for s in CFG['SEEDS']}

## 6 · Results (test set)

In [ ]:
def load_preds(seed, prompt):
    return pd.read_csv(os.path.join(RUN_DIR, f'seed_{seed}', f'test_preds_{prompt}.csv'))
METRICS = ['EM','F1','ROUGE_L','SemSim','NumRecall','Refusal_P','Refusal_R','Refusal_F1','FalseRefusal','DecisionAcc','TooShort','ContextSupport','AvgWords']

def table_for(key):
    seeds_m = pd.DataFrame([all_runs[s][key] for s in CFG['SEEDS']])
    mean = seeds_m[METRICS].mean()
    std = seeds_m[METRICS].std(ddof=1) if len(seeds_m) > 1 else mean * 0
    return pd.DataFrame({'Base': pd.Series({m: base_res[key][m] for m in METRICS}), 'LoRA mean': mean, 'LoRA std': std}).round(3)

n_orig = int((~test_df.synthetic).sum())
for prompt in ['train_prompt'] + (['deploy_prompt'] if CFG['EVAL_DEPLOY_PROMPT'] else []):
    print(f'\n=== {prompt} | NON-SYNTHETIC test rows (n={n_orig}) ===');   print(table_for(prompt + '_orig').to_string())
    print(f'\n=== {prompt} | AUGMENTED test set (n={len(test_df)}, refusal-aware) ===');  print(table_for(prompt).to_string())

nll = pd.DataFrame({'val': [base_res['answer_nll_val']] + [all_runs[s]['answer_nll_val'] for s in CFG['SEEDS']],
                    'test': [base_res['answer_nll_test']] + [all_runs[s]['answer_nll_test'] for s in CFG['SEEDS']]},
                   index=['base'] + [f'seed {s}' for s in CFG['SEEDS']])
nll['test_perplexity'] = np.exp(nll.test)
print('\nAnswer-only NLL / perplexity:'); print(nll.round(3).to_string())
print('\nSelected checkpoints:'); print(pd.DataFrame([{k: all_runs[s][k] for k in ['seed','best_step','stopped_step','sel_score','train_minutes']} for s in CFG['SEEDS']]).round(3).to_string(index=False))
for s in CFG['SEEDS']:
    print(f'\nSeed {s} checkpoint trace (validation selection set):'); print(pd.DataFrame(all_runs[s]['candidates']).round(3).to_string(index=False))

In [ ]:
# ── Statistical tests (paired bootstrap over test questions) ──────────────────
def per_example(prompt, metric, orig_only=False):
    arrs = []
    for s in CFG['SEEDS']:
        d = load_preds(s, prompt)
        arrs.append((d[~d.synthetic.astype(bool)] if orig_only else d)[metric].values)
    return np.mean(arrs, axis=0)
def base_example(prompt, metric, orig_only=False):
    d = pd.read_csv(os.path.join(base_dir, f'test_preds_{prompt}.csv'))
    return (d[~d.synthetic.astype(bool)] if orig_only else d)[metric].values

def paired_bootstrap(a, b, n=5000, seed=0):
    d = np.asarray(a) - np.asarray(b); rng = np.random.RandomState(seed)
    m = rng.choice(d, (n, len(d))).mean(1)
    return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5), max(2 * min((m <= 0).mean(), (m >= 0).mean()), 1/n)

rows = []
for prompt in ['train_prompt'] + (['deploy_prompt'] if CFG['EVAL_DEPLOY_PROMPT'] else []):
    for metric, orig in [('f1', True), ('rougeL', True), ('semsim', True), ('ooc_ok', False)]:
        lo_, ba_ = per_example(prompt, metric, orig), base_example(prompt, metric, orig)
        d, lo, hi, p = paired_bootstrap(lo_, ba_); ci = boot_ci(lo_)
        rows.append(dict(prompt=prompt, metric=metric + ('' if orig else ' (refuse/answer decision, aug. test)'), n=len(lo_),
                         base=ba_.mean(), lora=lo_.mean(), lora_95ci=f'[{ci[0]:.3f}, {ci[1]:.3f}]',
                         diff=d, diff_95ci=f'[{lo:.3f}, {hi:.3f}]', p=p))
print('Paired bootstrap, LoRA (seed-averaged) vs base:'); print(pd.DataFrame(rows).round(4).to_string(index=False))

# Per-category F1 + refusal-decision accuracy (augmented test set)
tp = pd.concat([load_preds(s, 'train_prompt').assign(seed=s) for s in CFG['SEEDS']])
cat_lora = tp.groupby('category').agg(n=('f1', 'size'), LoRA_F1=('f1', 'mean'), LoRA_decision=('ooc_ok', 'mean'))
cat_lora['n'] = cat_lora['n'] // len(CFG['SEEDS'])
b = pd.read_csv(os.path.join(base_dir, 'test_preds_train_prompt.csv'))
cat_base = b.groupby('category').agg(base_F1=('f1', 'mean'), base_decision=('ooc_ok', 'mean'))
percat = cat_lora.join(cat_base).round(3)[['n', 'base_F1', 'LoRA_F1', 'base_decision', 'LoRA_decision']]
percat['small_n(<5)'] = percat.n < 5
print('\nPer-category (augmented test set):'); print(percat.to_string())
percat.rename(columns={'LoRA_F1': 'LoRA_F1'}).to_csv(os.path.join(RUN_DIR, 'per_category_f1.csv'))

### Results by groundedness bucket
`in_passage`: reference answer is mostly *in* the passage (tests grounding / copying) · `beyond_passage`: reference needs knowledge not in the passage (tests memorised knowledge) · `computed`: derived from numbers in the question · `refusal`: correct answer is `OUT_OF_CONTEXT`.
Report **`in_passage` F1** as the grounding result; `beyond_passage` F1 mostly reflects what a 1.5 B model remembers.

In [ ]:
tp = pd.concat([load_preds(s, 'train_prompt') for s in CFG['SEEDS']])
bb = pd.read_csv(os.path.join(base_dir, 'test_preds_train_prompt.csv'))
g_l = tp.groupby('gbucket').agg(LoRA_F1=('f1', 'mean'), LoRA_SemSim=('semsim', 'mean'), LoRA_Support=('support', 'mean'), LoRA_decision=('ooc_ok', 'mean'))
g_b = bb.groupby('gbucket').agg(base_F1=('f1', 'mean'), base_SemSim=('semsim', 'mean'), base_Support=('support', 'mean'), base_decision=('ooc_ok', 'mean'))
gtab = g_b.join(g_l); gtab.insert(0, 'n', bb.groupby('gbucket').size())
print('Per groundedness bucket (augmented test set, train prompt):'); print(gtab.round(3).to_string())
gtab.to_csv(os.path.join(RUN_DIR, 'groundedness_buckets.csv'))
dp = pd.concat([load_preds(s, 'deploy_prompt') for s in CFG['SEEDS']]) if CFG['EVAL_DEPLOY_PROMPT'] else None
if dp is not None:
    print('\nSame, deployed system prompt (LoRA only):'); print(dp.groupby('gbucket').agg(F1=('f1', 'mean'), decision=('ooc_ok', 'mean')).round(3).to_string())

## 7 · Paper figures (IEEE column widths, vector PDF + PNG)

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
BLUE, ORANGE, GREEN, GREY, RED, PURPLE = '#0072B2', '#D55E00', '#009E73', '#6b6b6b', '#CC3311', '#8c5cb5'
plt.rcParams.update({'font.family':'serif','font.size':8,'axes.labelsize':8,'legend.fontsize':7,'xtick.labelsize':7,'ytick.labelsize':7,
                     'axes.linewidth':0.6,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.25,
                     'grid.linewidth':0.4,'pdf.fonttype':42,'savefig.dpi':300,'savefig.bbox':'tight'})
COL_W, FULL_W = 3.5, 7.16
FIG = os.path.join(RUN_DIR, 'figures')
def savefig(fig, name):
    for ext in ('pdf', 'png'): fig.savefig(os.path.join(FIG, f'{name}.{ext}'))
    plt.show()

# Fig 1: loss curves, mean +- std over seeds
logs = {s: json.load(open(os.path.join(RUN_DIR, f'seed_{s}', 'log_history.json'))) for s in CFG['SEEDS']}
def series(log, key): return pd.Series({h['step']: h[key] for h in log if key in h})
fig, ax = plt.subplots(figsize=(COL_W, 2.5))
for key, col, mk, ls, lab in [('loss', BLUE, 'o', '-', 'Training loss'), ('eval_loss', ORANGE, 's', '--', 'Validation loss')]:
    df = pd.DataFrame({s: series(logs[s], key) for s in CFG['SEEDS']}).dropna(how='all').sort_index()
    m, sd = df.mean(1), df.std(1).fillna(0)
    ax.plot(m.index, m.values, color=col, marker=mk, ms=3, lw=1.2, ls=ls, label=lab + (f' (mean of {len(CFG["SEEDS"])} seeds)' if len(CFG['SEEDS'])>1 else ''))
    ax.fill_between(m.index, m - sd, m + sd, color=col, alpha=0.18, lw=0)
bs_ = [all_runs[s]['best_step'] for s in CFG['SEEDS']]
ax.axvline(np.median(bs_), color=GREY, lw=0.8, ls=':'); ax.text(np.median(bs_), ax.get_ylim()[1]*0.95, ' median selected ckpt', fontsize=6.5, color=GREY, va='top')
ax.set_xlabel('Optimisation step'); ax.set_ylabel('Cross-entropy loss' + (' (answer tokens)' if CFG['TRAIN_ON_RESPONSES_ONLY'] else ''))
ax.legend(frameon=False); savefig(fig, 'fig_loss_curves')

# Fig 2: base vs LoRA headline metrics (augmented test set), clipped error bars
fig, ax = plt.subplots(figsize=(COL_W, 2.7))
names = ['F1', 'ROUGE_L', 'SemSim', 'NumRecall', 'Refusal_F1']; labs = ['Token F1', 'ROUGE-L', 'Semantic\nsim.', 'Number\nrecall', 'Refusal\nF1']
seeds_m = pd.DataFrame([all_runs[s]['train_prompt'] for s in CFG['SEEDS']])
mean = np.nan_to_num(seeds_m[names].mean().values.astype(float))
sdv = np.nan_to_num(seeds_m[names].std(ddof=1).values.astype(float)) if len(seeds_m) > 1 else np.zeros(len(names))
yerr = np.vstack([np.minimum(sdv, mean), np.minimum(sdv, 1 - mean)])
x = np.arange(len(names)); w = 0.38
ax.bar(x - w/2, np.nan_to_num([base_res['train_prompt'][n] for n in names]), w, color=GREY, label='Base (no fine-tune)')
ax.bar(x + w/2, mean, w, yerr=yerr, capsize=2, color=BLUE, hatch='//', edgecolor='white', label='QLoRA fine-tuned (mean ± sd)')
ax.set_xticks(x); ax.set_xticklabels(labs); ax.set_ylim(0, 1.05); ax.set_ylabel(f'Score (test, n={len(test_df)})')
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=6.5)
ax.grid(axis='x', visible=False); savefig(fig, 'fig_base_vs_lora')

# Fig 3: per-category F1
pc = percat.copy(); fig, ax = plt.subplots(figsize=(FULL_W, 2.5)); x = np.arange(len(pc)); w = 0.38
ax.bar(x - w/2, pc.base_F1, w, color=GREY, label='Base'); ax.bar(x + w/2, pc.LoRA_F1, w, color=BLUE, hatch='//', edgecolor='white', label='QLoRA')
ax.set_xticks(x); ax.set_xticklabels([f"{c.replace('_',' ')}\n(n={n})" for c, n in zip(pc.index, pc.n)], fontsize=6.5)
ax.set_ylim(0, 1.05); ax.set_ylabel('Token F1'); ax.legend(frameon=False); ax.grid(axis='x', visible=False); savefig(fig, 'fig_per_category_f1')

# Fig 4: train vs deployed system prompt (prompt-robustness)
if CFG['EVAL_DEPLOY_PROMPT']:
    fig, ax = plt.subplots(figsize=(COL_W, 2.4)); x = np.arange(3); w = 0.2
    nm = ['F1', 'Refusal_F1', 'FalseRefusal']
    def m(pr, who):
        if who == 'base': return [base_res[pr][n] for n in nm]
        d = pd.DataFrame([all_runs[s][pr] for s in CFG['SEEDS']]); return list(d[nm].mean())
    for i, (pr, who, col, hat, lab) in enumerate([('train_prompt','base',GREY,'','Base, train prompt'), ('deploy_prompt','base','#bdbdbd','..','Base, deploy prompt'),
                                                   ('train_prompt','lora',BLUE,'','LoRA, train prompt'), ('deploy_prompt','lora',ORANGE,'//','LoRA, deploy prompt')]):
        ax.bar(x + (i-1.5)*w, m(pr, who), w, color=col, hatch=hat, edgecolor='white', label=lab)
    ax.set_xticks(x); ax.set_xticklabels(['Token F1', 'Refusal F1', 'False-refusal\nrate']); ax.set_ylim(0, 1.15); ax.legend(frameon=False, fontsize=6, ncol=2)
    ax.grid(axis='x', visible=False); savefig(fig, 'fig_prompt_robustness')

# Fig 5: refusal confusion matrix (pooled over seeds, train prompt)
pool = pd.concat([load_preds(s, 'train_prompt') for s in CFG['SEEDS']])
cm = pd.crosstab(pool.ref_ooc, pool.pred_ooc).reindex(index=[False, True], columns=[False, True], fill_value=0)
fig, ax = plt.subplots(figsize=(2.4, 2.2)); ax.imshow(cm.values, cmap='Blues'); ax.grid(False)
for i in range(2):
    for j in range(2): ax.text(j, i, int(cm.values[i, j]), ha='center', va='center', color='white' if cm.values[i, j] > cm.values.max()/2 else 'black')
ax.set_xticks([0, 1]); ax.set_xticklabels(['answer', 'refuse']); ax.set_yticks([0, 1]); ax.set_yticklabels(['in-context', 'out-of-ctx'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Reference'); savefig(fig, 'fig_refusal_confusion')

# Fig 6: dataset composition after the group split
fig, ax = plt.subplots(figsize=(FULL_W, 2.5)); t2 = pd.crosstab(raw.category, raw.split).reindex(columns=['train','val','test']).fillna(0)
t2 = t2.loc[t2.sum(1).sort_values(ascending=False).index]; x = np.arange(len(t2)); w = 0.27
for i, (s, col) in enumerate([('train', BLUE), ('val', ORANGE), ('test', GREEN)]):
    ax.bar(x + (i-1)*w, t2[s].clip(lower=0.5), w, color=col, label=f'{s} (n={int(t2[s].sum())})')
    for xi, v in zip(x, t2[s]): ax.text(xi + (i-1)*w, max(v, .5)*1.1, int(v), ha='center', fontsize=6)
ax.set_yscale('log'); ax.set_ylim(0.4, 1500); ax.set_xticks(x); ax.set_xticklabels([c.replace('_', ' ') for c in t2.index], rotation=15)
ax.set_ylabel('Examples (log)'); ax.legend(frameon=False); ax.grid(axis='x', visible=False); savefig(fig, 'fig_dataset_split')
# Fig 7: why generation-based checkpoint selection (mean over seeds)
cd = pd.concat([pd.DataFrame(all_runs[s]['candidates']).assign(seed=s) for s in CFG['SEEDS']])
if CFG['SELECT_BY'] == 'generation':
    g = cd.groupby('step')[['score', 'F1', 'Refusal_F1', 'val_loss']].mean()
    fig, ax = plt.subplots(figsize=(COL_W, 2.5)); ax2 = ax.twinx(); ax2.grid(False); ax2.spines['right'].set_visible(True)
    ax.plot(g.index, g.F1, color=BLUE, marker='o', ms=3, lw=1.2, label='Val F1')
    ax.plot(g.index, g.Refusal_F1, color=ORANGE, marker='s', ms=3, lw=1.2, ls='--', label='Val refusal F1')
    ax2.plot(g.index, g.val_loss, color=GREY, marker='^', ms=3, lw=1, ls=':', label='Val loss')
    ax.set_xlabel('Checkpoint (optimisation step)'); ax.set_ylabel('Score'); ax2.set_ylabel('Validation loss')
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, frameon=False, fontsize=6.5, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=3)
    savefig(fig, 'fig_checkpoint_selection')
# Fig 8: F1 / decision accuracy by groundedness bucket
order = [b for b in ['in_passage', 'beyond_passage', 'computed', 'refusal'] if b in gtab.index]
fig, (a, b2) = plt.subplots(1, 2, figsize=(FULL_W, 2.4)); x = np.arange(len(order)); w = 0.38
a.bar(x - w/2, gtab.loc[order, 'base_F1'], w, color=GREY, label='Base'); a.bar(x + w/2, gtab.loc[order, 'LoRA_F1'], w, color=BLUE, hatch='//', edgecolor='white', label='QLoRA')
a.set_xticks(x); a.set_xticklabels([f"{o.replace('_', ' ')}\n(n={int(gtab.loc[o, 'n'])})" for o in order], fontsize=6.5); a.set_ylabel('Token F1'); a.set_ylim(0, 1.05)
a.set_title('(a) F1 by where the answer lives', fontsize=8); a.legend(frameon=False); a.grid(axis='x', visible=False)
b2.bar(x - w/2, gtab.loc[order, 'base_decision'], w, color=GREY); b2.bar(x + w/2, gtab.loc[order, 'LoRA_decision'], w, color=BLUE, hatch='//', edgecolor='white')
b2.set_xticks(x); b2.set_xticklabels([o.replace('_', ' ') for o in order], fontsize=6.5); b2.set_ylabel('Answer / refuse decision accuracy'); b2.set_ylim(0, 1.05)
b2.set_title('(b) Refuse-vs-answer decision', fontsize=8); b2.grid(axis='x', visible=False)
fig.tight_layout(); savefig(fig, 'fig_groundedness')
print('Figures saved to', FIG)

## 8 · Error analysis (qualitative table for the paper)

In [ ]:
best_seed = max(CFG['SEEDS'], key=lambda s: all_runs[s]['sel_score'])     # chosen on VALIDATION, never on test
print('Best seed (highest validation selection score):', best_seed)
bp = load_preds(best_seed, 'train_prompt')
bp['verdict'] = np.select([bp.ref_ooc & bp.pred_ooc, bp.ref_ooc & ~bp.pred_ooc, ~bp.ref_ooc & bp.pred_ooc],
                          ['correct refusal', 'MISSED refusal (hallucinated)', 'FALSE refusal'], default='answered')
print(bp.verdict.value_counts().to_string())
pd.set_option('display.max_colwidth', 160)
worst = bp[~bp.ref_ooc].sort_values('f1').head(5)[['category','question','completion','pred','f1']]
best  = bp[~bp.ref_ooc].sort_values('f1', ascending=False).head(3)[['category','question','completion','pred','f1']]
print('\n--- 5 lowest-F1 answers ---'); print(worst.to_string(index=False))
print('\n--- 3 highest-F1 answers ---'); print(best.to_string(index=False))
bp.to_csv(os.path.join(RUN_DIR, 'error_analysis_best_seed.csv'), index=False)

## 9 · Export: best adapter → merged fp16 model → sanity check → zip

In [ ]:
best_adapter = os.path.join(RUN_DIR, f'seed_{best_seed}', 'adapter')
MERGE_DIR = os.path.join(RUN_DIR, 'merged_fp16')
if not os.path.exists(os.path.join(MERGE_DIR, 'config.json')):
    model, tok = FastLanguageModel.from_pretrained(model_name=best_adapter, max_seq_length=CFG['MAX_SEQ_LEN'], load_in_4bit=True, dtype=None)
    model.save_pretrained_merged(MERGE_DIR, tok, save_method='merged_16bit')
    free_gpu(model, tok)
print('Merged model:', MERGE_DIR)

# Sanity check: the merged fp16 model should score about the same as adapter+4bit on a test subset.
# (Not bit-identical: the adapter run uses a 4-bit base, the merged model is fp16.)
from transformers import AutoModelForCausalLM
sub = test_df.head(24)
mtok = AutoTokenizer.from_pretrained(MERGE_DIR)
mm = AutoModelForCausalLM.from_pretrained(MERGE_DIR, torch_dtype=torch.float16, device_map='cuda')
msc = score_predictions(sub, generate_answers(mm, mtok, sub))
ref = load_preds(best_seed, 'train_prompt').head(24)
print(f"Merged fp16  F1={msc.f1.mean():.3f} | adapter+4bit F1={ref.f1.mean():.3f} | identical outputs: {(msc.pred.values == ref.pred.values).mean():.0%}")
free_gpu(mm, mtok)

art = os.path.join(RUN_DIR, 'artifacts'); os.makedirs(art, exist_ok=True)
shutil.copytree(best_adapter, os.path.join(art, 'best_adapter'), dirs_exist_ok=True)
shutil.copytree(os.path.join(RUN_DIR, 'figures'), os.path.join(art, 'figures'), dirs_exist_ok=True)
for f in ['config.json', 'data_audit.json', 'per_category_f1.csv', 'error_analysis_best_seed.csv']:
    shutil.copy(os.path.join(RUN_DIR, f), art)
json.dump({'env': ENV, 'best_seed': best_seed, 'runs': all_runs, 'base': base_res}, open(os.path.join(art, 'results.json'), 'w'), indent=2, default=str)
zip_path = shutil.make_archive(os.path.join(WORK_DIR, f"os_tutor_ft2_{CFG['EXPERIMENT']}"), 'zip', art)
print('Artifacts zip:', zip_path)
if IN_COLAB:
    from google.colab import files
    shutil.copy(zip_path, '/content/' + os.path.basename(zip_path))
    try: files.download('/content/' + os.path.basename(zip_path))
    except Exception as e: print('Browser download failed (', e, ') -> get the zip from Drive:', zip_path)

## 10 · Inference demo (format must match training)

In [ ]:
model, tok = FastLanguageModel.from_pretrained(model_name=best_adapter, max_seq_length=CFG['MAX_SEQ_LEN'], load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(model)

def ask(context, question, system=SYSTEM_PROMPT):
    # EXACT user-turn format used in training: 'Textbook passages:\n<ctx>\n\nQuestion: <q>'
    user = f'Textbook passages:\n{context}\n\nQuestion: {question}'
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    tok.padding_side = 'left'
    enc = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=CFG['GEN_MAX_NEW_TOKENS'], do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()

ctx = ('A deadlock is a situation where a set of processes are blocked because each process is holding a resource '
       'and waiting for another resource acquired by some other process.')
print('Q: What is a deadlock?\nA:', ask(ctx, 'What is a deadlock?'))
print('\nQ: What is the capital of France?  (expect OUT_OF_CONTEXT)\nA:', ask(ctx, 'What is the capital of France?'))
ex = test_df[~test_df.category.isin(['out_of_context'])].iloc[0]
print(f'\n[test example, {ex.category}]\nQ: {ex.question[:200]}\nA: {ask(ex.context, ex.question)[:500]}')
free_gpu(model, tok)

## 11 · Ablation study — compare experiments
Set `PRESET` in the Config cell (e.g. `abl_no_aug`), **restart the runtime**, *Run all* (use `SEEDS=[42]` in a preset if time is short). Each preset writes its own folder, and the next cell tabulates all of them.

| `PRESET` | Change | Question |
|---|---|---|
| `main` | defaults | the full v3 method |
| `abl_no_aug` | no off-topic / hard-negative rows | does refusal augmentation fix refusal? |
| `abl_no_promptmix` | `PROMPT_MIX=0` | does it fix the deployed-prompt gap? |
| `abl_select_by_loss` | pick checkpoint by val loss | v2-style selection vs generation-based |
| `abl_fullseq_loss` | loss on the whole sequence | does answer-only loss matter? |
| `abl_r8` / `abl_r32` | LoRA rank | rank sensitivity |

In [ ]:
rows = []
for exp in sorted(os.listdir(WORK_DIR)):
    d = os.path.join(WORK_DIR, exp)
    if not os.path.isdir(d) or not os.path.exists(os.path.join(d, 'config.json')): continue
    ms = [json.load(open(os.path.join(d, s, 'metrics.json'))) for s in sorted(os.listdir(d)) if s.startswith('seed_') and os.path.exists(os.path.join(d, s, 'metrics.json'))]
    if not ms: continue
    def pick(m, k, key): return m.get(k, m['train_prompt'])[key]
    df = pd.DataFrame([{'F1_orig': pick(m, 'train_prompt_orig', 'F1'), 'SemSim_orig': pick(m, 'train_prompt_orig', 'SemSim') if 'SemSim' in m['train_prompt'] else np.nan,
                        'RefusalF1': m['train_prompt']['Refusal_F1'], 'FalseRefusal': m['train_prompt']['FalseRefusal'],
                        'F1_deploy_orig': pick(m, 'deploy_prompt_orig', 'F1') if 'deploy_prompt' in m else np.nan,
                        'RefusalF1_deploy': m['deploy_prompt']['Refusal_F1'] if 'deploy_prompt' in m else np.nan,
                        'test_NLL': m['answer_nll_test']} for m in ms])
    f = lambda c: ('n/a' if df[c].isna().all() else f'{df[c].mean():.3f}' + (f' ± {df[c].std(ddof=1):.3f}' if len(df) > 1 else ''))
    rows.append({'experiment': exp, 'seeds': len(ms), **{c: f(c) for c in df.columns}})
print(pd.DataFrame(rows).to_string(index=False) if rows else 'No finished experiments yet.')

## 12 · Reporting checklist (what this notebook still does *not* do for you)
* **Best fix = real passages.** The lexical `support` test is a proxy. Re-retrieve passages for every non-numerical question from your Qdrant textbook index, keep rows whose retrieved text supports the answer, relabel the rest `OUT_OF_CONTEXT`, and save as `dataset_v3.csv` (same columns); the notebook picks it up automatically.
* **Review `relabeled_for_review.csv`.** Any relabelled row that *is* answerable from its passage is label noise; check a sample (say 30) and report the error rate in the paper.
* **Synthetic / relabelled refusals are not new knowledge.** Describe how they were built (§2, §2b). Treat the original-row results as primary and the augmented test set as a refusal stress test.
* **Reasoning categories** (analytical, conceptual, comparative, edge-case) have ≈ 2 test rows each — anecdotal. They also need *fuller* reference answers; `TooShort` tracks the one-word-answer failure.
* **Passages are generic and reused** (126 unique for 762 rows): results show generalisation to *unseen passages*, not unseen topics or the real textbook.
* **End-to-end RAG** (RAGAS on the full question set, not 5) lives in `eval/`; this notebook scores the generator with gold passages. Report both, and serve with the prompt that matches your evaluation.
* **Human evaluation** (≥ 3 raters, ≥ 50 answers, κ) is still the most-requested missing item for an education paper.
* Library versions are printed in §1 and saved in `results.json`; cite them.